In [ ]:
%cd /kaggle/working
!rm -rf BasicSR Real-ESRGAN
!git clone https://github.com/XPixelGroup/BasicSR.git
!git clone https://github.com/xinntao/Real-ESRGAN.git

from pathlib import Path

setup_py = Path('/kaggle/working/BasicSR/setup.py')
source = setup_py.read_text()
old = "exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
new = "ns = {}\n        exec(compile(f.read(), version_file, 'exec'), ns)\n    return ns['__version__']"
assert old in source, 'BasicSR setup.py version pattern not found'
setup_py.write_text(source.replace(old, new))

# Patch before installing so the installed package gets the fix too.
degradations = Path('/kaggle/working/BasicSR/basicsr/data/degradations.py')
assert degradations.exists(), 'basicsr degradations.py not found'
degradations.write_text(
    degradations.read_text().replace(
        'torchvision.transforms.functional_tensor',
        'torchvision.transforms.functional',
    )
)

!pip install -q --no-deps --no-build-isolation ./BasicSR
!pip install -q addict yapf lmdb ffmpeg-python
!pip install -q realesrgan --no-deps

version_file = Path('/kaggle/working/Real-ESRGAN/realesrgan/version.py')
version_file.write_text("__version__ = '0.3.0'\n__gitsha__ = 'unknown'\nversion_info = (0, 3, 0)\n")

%cd /kaggle/working/Real-ESRGAN
from realesrgan import RealESRGANer
print('SETUP DONE')

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    os.environ['HEROKU_URL'] = client.get_secret('HEROKU_URL')
    os.environ['WORKER_SECRET'] = client.get_secret('WORKER_SECRET')
except Exception as exc:
    print(f'Kaggle secrets unavailable: {exc}', flush=True)

H = os.environ.get('HEROKU_URL', '').rstrip('/')
WORKER_SECRET = os.environ.get('WORKER_SECRET', '')
HDR = {'x-worker-secret': WORKER_SECRET}

POLL = 5
HB = 15
MODEL = 'realesr-general-x4v3'
SCALE = 4
MAX_DUR = 30

IN_PATH = '/kaggle/working/in.mp4'
OUT_DIR = '/kaggle/working/out_dir'
OUT_PATH = f'{OUT_DIR}/in_out.mp4'

state = {'busy': False, 'jobId': None}


def gpu_info():
    try:
        import torch
        if not torch.cuda.is_available():
            return {'available': False, 'name': None, 'vram_mb': None, 'model': MODEL}
        props = torch.cuda.get_device_properties(torch.cuda.current_device())
        return {
            'available': True,
            'name': props.name,
            'vram_mb': int(props.total_memory / (1024 * 1024)),
            'model': MODEL,
        }
    except Exception:
        return {'available': False, 'name': None, 'vram_mb': None, 'model': MODEL}


print('Worker config ready for', H or '(missing HEROKU_URL)')

In [ ]:
import threading
import time

import requests

last_error = None


def heartbeat_loop():
    global last_error
    while True:
        try:
            if not H or not WORKER_SECRET:
                raise RuntimeError('HEROKU_URL or WORKER_SECRET is missing')
            r = requests.post(f'{H}/api/worker/heartbeat', json=state, headers=HDR, timeout=20)
            r.raise_for_status()
            ping_id = r.json().get('pingId')
            if ping_id:
                p = requests.post(
                    f'{H}/api/worker/ping/{ping_id}',
                    json={**gpu_info(), **state},
                    headers=HDR,
                    timeout=20,
                )
                p.raise_for_status()
            message = None
        except Exception as exc:
            message = str(exc)
        if message != last_error:
            if message:
                print('heartbeat error:', message, flush=True)
            last_error = message
        time.sleep(HB)


threading.Thread(target=heartbeat_loop, daemon=True).start()
print('Heartbeat thread started')

In [ ]:
import os
import subprocess
import time

import requests


def auth_headers(url):
    return dict(HDR) if url == H or url.startswith(H + '/') else {}


def absolute(url):
    if url.startswith('http://') or url.startswith('https://'):
        return url
    return f'{H}{url}'


def video_duration(path):
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', path],
        capture_output=True, text=True, timeout=30,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'ffprobe failed')
    return float(result.stdout.strip())


while True:
    job = None
    started = None
    try:
        r = requests.get(f'{H}/api/worker/next', headers=HDR, timeout=30)
        r.raise_for_status()
        job = r.json().get('job')
        if not job:
            time.sleep(POLL)
            continue

        state['busy'] = True
        state['jobId'] = job['jobId']
        started = time.time()
        print(f"[{job['jobId']}] start", flush=True)

        input_url = absolute(job['inputUrl'])
        with requests.get(input_url, headers=auth_headers(input_url), timeout=120, stream=True) as resp:
            resp.raise_for_status()
            with open(IN_PATH, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

        duration = video_duration(IN_PATH)
        # NO DURATION LIMIT - Process any length video
        print(f"[{job['jobId']}] video duration: {duration:.1f}s - processing...", flush=True)

        os.makedirs(OUT_DIR, exist_ok=True)
        result = subprocess.run(
            [
                'python', 'inference_realesrgan_video.py',
                '-n', MODEL,
                '-i', IN_PATH,
                '-o', OUT_DIR,
                '-s', str(SCALE),
                '--suffix', 'out',
            ],
            cwd='/kaggle/working/Real-ESRGAN',
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError((result.stderr or result.stdout or 'inference failed').strip()[-300:])

        if not os.path.exists(OUT_PATH):
            raise FileNotFoundError(f'expected output not found: {OUT_PATH}')

        output_url = absolute(job['outputUrl'])
        headers = dict(auth_headers(output_url))
        headers['Content-Type'] = 'video/mp4'
        with open(OUT_PATH, 'rb') as f:
            payload = f.read()
        put = requests.put(output_url, data=payload, headers=headers, timeout=300)
        if put.status_code == 403:
            retry_headers = {k: v for k, v in headers.items() if k.lower() != 'content-type'}
            put = requests.put(output_url, data=payload, headers=retry_headers, timeout=300)
        put.raise_for_status()

        elapsed = time.time() - started
        done = requests.post(
            f"{H}/api/worker/{job['jobId']}/done",
            json={'processingSeconds': round(elapsed, 2)},
            headers=HDR,
            timeout=30,
        )
        done.raise_for_status()
        print(f"[{job['jobId']}] DONE in {elapsed:.1f}s", flush=True)
    except Exception as exc:
        elapsed = time.time() - started if started else 0
        if job and job.get('jobId'):
            try:
                requests.post(
                    f"{H}/api/worker/{job['jobId']}/failed",
                    json={'message': str(exc)[:300]},
                    headers=HDR,
                    timeout=30,
                )
            except Exception as report_exc:
                print(f"[{job.get('jobId')}] failed to report error: {report_exc}", flush=True)
        print(f'job failed after {elapsed:.1f}s: {exc}', flush=True)
    finally:
        state['busy'] = False
        state['jobId'] = None
        time.sleep(POLL)